**Notebook to explore Research Question 2** 

Does static typing reduce maintenance burden for AI-generated code relative to dynamically typed languages? 

Core Hypothesis:

The maintenance burden gap between AI-generated and human-generated pull requests is smaller in statically typed languages than in dynamically typed languages.

**Dataset Loading & Imports** 

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [2]:
df = pd.read_parquet('../results/rq1_main_frame.parquet')
df.head()

,id,number,title,body,agent,user_id,user,state,created_at,closed_at,...,pr_outcome,fix_resolution_time,fix_iteration_count,pr_size_loc,pr_additions,pr_deletions,fix_size,defect_count_90d,has_defect_90d,defect_density
0,3264933329,2911,Fix: Wait for all partitions in load_collectio...,## Summary\n\nFixes an issue where `load_colle...,Claude_Code,108661493,weiliu1031,closed,2025-07-26 02:59:01+00:00,2025-07-29 07:01:20+00:00,...,rejected,NaN,NaN,396.0,394.0,2.0,396.0,0,0,0.000000
1,3265640341,30,Add build staleness detection for debug CLI,## Summary\r\n\r\n Implements comprehensive b...,Claude_Code,7475,MSch,closed,2025-07-26 13:31:19+00:00,2025-07-26 13:37:22+00:00,...,accepted,NaN,NaN,407.0,298.0,109.0,NaN,1,1,0.002457
2,3265782173,17625,chore: remove HashedPostStateProvider trait,## Summary\r\n\r\n#17545 \r\n\r\nRemove the un...,Claude_Code,47593288,adust09,open,2025-07-26 15:02:48+00:00,NaT,...,open,NaN,NaN,221.0,53.0,168.0,NaN,0,0,0.000000
3,3231949586,32656,feat(swagger): Add Swagger annotations to Batc...,## Summary\nProgressive implementation of Swag...,Claude_Code,1236198,spbolton,open,2025-07-15 11:46:41+00:00,NaT,...,open,NaN,NaN,3247.0,2702.0,545.0,NaN,0,0,0.000000
4,3231950376,32657,feat(swagger): Add Swagger annotations to Batc...,## Summary\nProgressive implementation of Swag...,Claude_Code,1236198,spbolton,open,2025-07-15 11:46:57+00:00,NaT,...,open,NaN,NaN,13888.0,12061.0,1827.0,NaN,0,0,0.000000


In [3]:
# Keep only static and dynamic
df_rq2 = df[df["language_type_group"].isin([0, 1])].copy()

# Create binary indicator: 1 = static, 0 = dynamic
df_rq2["is_static"] = (df_rq2["language_type_group"] == 0).astype(int)

print("Rows remaining:", len(df_rq2))
print(df_rq2["is_static"].value_counts())


Rows remaining: 17579
is_static
1    12497
0     5082
Name: count, dtype: int64


In [4]:
df_rq2["log_pr_size"] = np.log(df_rq2["pr_size_loc"] + 1)
df_rq2["log_stars"]   = np.log(df_rq2["repo_stars"] + 1)


In [5]:
needed = [
    "has_defect_90d",
    "defect_count_90d",
    "ai_pr",
    "is_static",
    "log_pr_size",
    "repo_age_years",
    "log_stars",
    "contributor_count",
    "task_type_group",
    "repo_full_name"
]

df_rq2 = df_rq2[needed].dropna().copy()

groups = df_rq2["repo_full_name"].to_numpy()

print("Final RQ2 rows:", len(df_rq2))


Final RQ2 rows: 14878


**Does static typing reduce maintenance burden for AI PRs?**

In [6]:
df_rq2.groupby(["is_static", "ai_pr"])["has_defect_90d"].mean().unstack()

ai_pr,0,1
is_static,,
0,0.319018,0.198944
1,0.280108,0.201707


Interpretation:

AI reduces defects in both dynamic and static.

The reduction is slightly larger in dynamic.

Static languages do not dramatically amplify AI advantage.

In [7]:

rq2_logit = smf.logit(
    "has_defect_90d ~ ai_pr * is_static + log_pr_size + repo_age_years + log_stars + contributor_count + C(task_type_group)",
    data=df_rq2
).fit(disp=0, cov_type="cluster", cov_kwds={"groups": groups})

print(rq2_logit.summary())


                           Logit Regression Results                           
Dep. Variable:         has_defect_90d   No. Observations:                14878
Model:                          Logit   Df Residuals:                    14866
Method:                           MLE   Df Model:                           11
Date:                Wed, 11 Feb 2026   Pseudo R-squ.:                 0.05309
Time:                        18:32:00   Log-Likelihood:                -7568.7
converged:                       True   LL-Null:                       -7993.1
Covariance Type:              cluster   LLR p-value:                6.546e-175
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -0.5831      0.470     -1.241      0.215      -1.504       0.338
C(task_type_group)[T.1.0]     0.1748      0.089      1.974      0.048       0.001     

The AI advantage does NOT significantly change between static and dynamic languages.

In [8]:

rq2_nb = smf.glm(
    "defect_count_90d ~ ai_pr * is_static + log_pr_size + repo_age_years + log_stars + contributor_count + C(task_type_group)",
    data=df_rq2,
    family=sm.families.NegativeBinomial()
).fit(cov_type="cluster", cov_kwds={"groups": groups})

print(rq2_nb.summary())


/Users/stevotrujillo/Desktop/CS260-MBILLMC/.venv/lib/python3.9/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:       defect_count_90d   No. Observations:                14878
Model:                            GLM   Df Residuals:                    14866
Model Family:        NegativeBinomial   Df Model:                           11
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -17587.
Date:                Wed, 11 Feb 2026   Deviance:                       20517.
Time:                        18:32:29   Pearson chi2:                 4.54e+04
No. Iterations:                    13   Pseudo R-squ. (CS):             0.4099
Covariance Type:              cluster                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

Static typing does not significantly moderate the AI effect on defect counts.

In [9]:

print("Logit Odds Ratios:")
print(np.exp(rq2_logit.params))

print("\nNB Rate Ratios:")
print(np.exp(rq2_nb.params))


Logit Odds Ratios:
Intercept                    0.558142
C(task_type_group)[T.1.0]    1.190997
C(task_type_group)[T.2.0]    0.903304
C(task_type_group)[T.3.0]    1.480527
C(task_type_group)[T.4.0]    0.899722
ai_pr                        0.501267
is_static                    1.073835
ai_pr:is_static              1.227707
log_pr_size                  1.164643
repo_age_years               0.946287
log_stars                    0.869451
contributor_count            1.012621
dtype: float64

NB Rate Ratios:
Intercept                    0.555458
C(task_type_group)[T.1.0]    1.476395
C(task_type_group)[T.2.0]    0.631067
C(task_type_group)[T.3.0]    1.233355
C(task_type_group)[T.4.0]    0.806868
ai_pr                        0.631842
is_static                    1.450587
ai_pr:is_static              0.955733
log_pr_size                  1.316698
repo_age_years               0.912373
log_stars                    0.864781
contributor_count            1.025667
dtype: float64


While AI-generated PRs exhibit lower short-term corrective maintenance burden overall, this effect does not significantly differ between statically and dynamically typed languages. This suggests that static typing constraints do not materially amplify or attenuate the maintenance characteristics of AI-generated code.